# se-dat walkthrough — Titanic dataset

`se-dat` (Simple Exploratory Data Analysis) takes a raw DataFrame and gives you:

1. **Column type profiling** — inferred semantic types, missing %, cardinality, confidence flags.
2. **Correlation analysis** — Pearson/Spearman (numeric), Cramér's V (categorical), correlation ratio η (mixed), heatmaps, multicollinearity flags.
3. **Encoding suggestions** — binary maps for `yes/no` style columns, one-hot for low-cardinality categoricals, target/ordinal encoding for high-cardinality ones.
4. **Missing-value handling** — per-column imputation plans (median / mode / drop).
5. **Outlier detection** — IQR and z-score flags surfaced right in the profile table.
6. **Artifacts & interop** — static one-file HTML reports and scikit-learn `ColumnTransformer`s.

Available on PyPI: [`pip install se-dat`](https://pypi.org/project/se-dat/)

This notebook runs the whole pipeline on the classic Titanic passenger data.

In [ ]:
# If you haven't installed the package yet, uncomment:
# %pip install se-dat

## 1. Load the data

In [ ]:
import pandas as pd
import sedat

print("se-dat version:", sedat.__version__)

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)
df.head()

In [ ]:
print(f"{df.shape[0]} rows x {df.shape[1]} columns")
df.describe(include="all").T

What we already know going in:

- `Survived` (0/1) is our prediction target.
- `Age` has missing values, `Cabin` is mostly missing.
- `Sex` and `Embarked` are strings, `Pclass` is a number but really an ordinal category,
- `Name`, `Ticket`, `PassengerId` are high-cardinality text/ids.

Let's see how much of that `se-dat` recovers automatically.

## 2. One-call report: `EDAReport.create`

The facade runs profiling, correlations and encoding suggestions in one shot.
Passing `target=` enables target-encoding suggestions for high-cardinality categoricals.

In [ ]:
report = sedat.EDAReport.create(df, target=df["Survived"])

In [ ]:
report.profile.summary

Things to notice in the profile:

- `PassengerId` was recognized as an **id** column (unique integers).
- `Survived` (int 0/1) is flagged **boolean** with *medium* confidence — 0/1 is ambiguous between a flag and a real number.
- `Age` shows its **missing_pct**, `Cabin` shows ~77% missing.
- `Sex`, `Embarked` land in **categorical**; `Name`/`Ticket` in **string** (high-cardinality free text).- The **outliers** column notes numeric columns with values outside IQR fences — `Fare` has a handful of extreme ticket prices.

In [ ]:
report.correlations.flagged_pairs

In [ ]:
report.encoding_summary

## 3. Profiling, piece by piece

Everything in the report is also available as standalone functions.

In [ ]:
profile = sedat.profile_dataframe(df)
profile.summary

In [ ]:
# Inspect inference on individual, tricky columns
for col in ["PassengerId", "Survived", "Cabin"]:
    print(f"--- {col} ---")
    print(sedat.infer_column_type(df[col]), end="\n\n")

## 4. Correlation analysis

`se-dat` covers three pairings:

| pairing | measure | function |
|---|---|---|
| numeric ↔ numeric | Pearson / Spearman | `numeric_correlation` |
| categorical ↔ categorical | Cramér's V | `categorical_correlation` |
| numeric ↔ categorical | correlation ratio η | `numeric_categorical_correlation` |

In [ ]:
pearson = sedat.numeric_correlation(df, method="pearson")
pearson

In [ ]:
fig = sedat.correlation_heatmap(pearson, title="Pearson correlation (numeric columns)");

### Spearman (rank-based)

`Pclass` vs `Fare` is stronger under Spearman than Pearson — the fare distribution is heavily skewed, so ranks tell a cleaner story.

In [ ]:
spearman = sedat.numeric_correlation(df, method="spearman")
fig = sedat.correlation_heatmap(spearman, title="Spearman correlation (rank-based)");

### Cramér's V (categorical ↔ categorical)

> Caveat: Cramér's V is biased upward for high-cardinality columns, so take the `Ticket`/`Cabin` numbers with a grain of salt — `Sex` vs `Embarked` is the meaningful cell here.

In [ ]:
cramers = sedat.categorical_correlation(df)
fig = sedat.correlation_heatmap(cramers, title="Cramér's V (categorical columns)");

In [ ]:
# Correlation ratio: how well does each categorical column explain each numeric column?
eta = sedat.numeric_categorical_correlation(df)
eta.round(3)

In [ ]:
fig = sedat.correlation_heatmap(eta, title="Correlation ratio η (rows: numeric, cols: categorical)");

In [ ]:
# Pairs above the |0.7| threshold — potential multicollinearity
sedat.correlation_report(df, threshold=0.7).flagged_pairs

## 5. Encoding suggestions

Strategies chosen per column:

| situation | strategy |
|---|---|
| binary-like strings (`yes/no`) | `binary_encode` → 0/1 |
| categorical with ≤ `cardinality_threshold` values | `one_hot` |
| high-cardinality categorical **with** target | `target_encode` |
| high-cardinality categorical **without** target | `ordinal_encode` (warns) |

On Titanic that means: `Sex` → binary map, `Embarked` → one-hot. Everything else is left alone.

In [ ]:
plan = sedat.suggest_encodings(df, target=df["Survived"], cardinality_threshold=10)
plan.summary

### Applying a single suggestion

You don't have to accept everything — pick suggestions individually:

In [ ]:
sex_step = next(s for s in plan.suggestions if s.column == "Sex")
step_df = sex_step.apply(df)

step_df[["Sex", "Embarked"]].head()

### Applying everything at once

`apply_all_encodings` returns a new frame — the original `df` is never mutated.

In [ ]:
model_ready = report.apply_all_encodings()
model_ready.dtypes.to_frame("dtype")

In [ ]:
model_ready.head()

## 6. Missing-value handling

`suggest_imputations` returns an `ImputationPlan` with one suggestion per incomplete column:

- **numeric** -> median fill (the rationale mentions the mean as an alternative),
- **categorical/boolean-like** -> mode fill,
- anything missing above a configurable threshold (default 50%) -> **drop the column** instead of inventing data.

On Titanic that means: `Age` gets the median, `Embarked` gets the mode, and `Cabin` (~77% missing) is recommended for dropping.

In [ ]:
imp_plan = sedat.suggest_imputations(df)
imp_plan.summary

In [ ]:
# one column at a time...
filled = imp_plan.apply(df, column="Age")
print("Age median used:", imp_plan.get("Age").fill_value)

# ...or everything at once — also available as report.apply_all_imputations()
complete = report.apply_all_imputations()
complete.isna().sum().to_frame("missing_after")

## 7. Outlier detection

`detect_outliers` works over numeric columns with two methods:
`"iqr"` (default, values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`) and
`"zscore"` (|z| > 3). It reports per-column counts and percentages plus the
flagged row labels. `EDAReport.create` already runs it for you and shows a
one-line note per column in `report.profile.summary`.

In [ ]:
outliers = sedat.detect_outliers(df, method="iqr", iqr_scale=1.5)
outliers.summary.round(2)

In [ ]:
fare = outliers.get("Fare")
print(f"Fare fences: ({fare.lower:.2f}, {fare.upper:.2f})")
df.loc[fare.indices, ["Fare"]].head()

## 8. One-file HTML report

`report.to_html(path)` writes a static, self-contained page — every summary
table plus the correlation heatmaps embedded as base64 PNGs. Inline CSS only,
no JavaScript, no external assets: easy to e-mail or drop next to your repo.

In [ ]:
import os

out_path = report.to_html("titanic_eda_report.html")
print(f"wrote {out_path} ({os.path.getsize(out_path):,} bytes)")

# To view it inside Jupyter:
# from IPython.display import IFrame
# IFrame(out_path, width=1000, height=600)

## 9. scikit-learn interop

`EncodingPlan.to_column_transformer()` mirrors the suggested encodings with
native sklearn transformers — `OrdinalEncoder` for binary maps,
`OneHotEncoder` for low-cardinality categoricals, `TargetEncoder` for
high-cardinality ones — while already-numeric columns pass through.

The payoff vs. applying encodings by hand: fit on a training frame, then
`.transform()` a test frame and unseen categories are handled gracefully
(all-zeros / `-1`) instead of crashing or changing the output shape.
Requires the optional extra: `%pip install se-dat[sklearn]`.

In [ ]:
try:
    ct = plan.to_column_transformer()
except ImportError as exc:
    print(exc)  # scikit-learn not installed
else:
    features = df.drop(columns=["Survived"])
    Xt = ct.fit_transform(features, df["Survived"])
    print(f"transformer: {type(ct).__name__}")
    transformed = pd.DataFrame(Xt, columns=ct.get_feature_names_out())
    transformed.head()

## 10. Datetime feature extraction

`suggest_datetime_features` proposes calendar features for every detected
datetime column: year, month, day, day-of-week (`_dow`, Monday=0),
`_is_weekend` — plus `_hour` only when the data actually contains a time
component. The Titanic CSV has no date column, so we demo on a small
synthetic one:

In [ ]:
flights = pd.DataFrame(
    {
        "departure": pd.to_datetime(
            [
                "2024-01-06 08:30",   # Saturday
                "2024-01-08 19:45",   # Monday
                "2024-02-10 06:00",
                None,
            ]
        )
    }
)

dt_plan = sedat.suggest_datetime_features(flights)
dt_plan.summary

In [ ]:
featurized = sedat.apply_datetime_features(flights)
# nullable Int64 columns: missing timestamps stay <NA> instead of sentinels
featurized[["departure", "departure_dow", "departure_is_weekend", "departure_hour"]]

## 11. Save & load plans

Plans serialize their *decisions* (categories, fill values, code mappings) to
JSON — not function references — so a plan fitted on training data can be
re-applied to new data later with identical semantics: stable one-hot schema,
frozen binary/ordinal codes, stored target means.

We fit on the first 700 passengers, then transform a test slice that
deliberately lacks one `Embarked` category.

In [ ]:
train_df = df.iloc[:700]
train_plan = sedat.suggest_encodings(train_df, target=train_df["Survived"])
train_plan.save("titanic_encoding_plan.json")

# imputation and datetime plans expose the same .save()/.load() API:
# sedat.suggest_imputations(df).save("imputation_plan.json")
# sedat.EncodingPlan.load / sedat.ImputationPlan.load / sedat.DatetimeFeaturePlan.load

In [ ]:
loaded = sedat.EncodingPlan.load("titanic_encoding_plan.json")

test_df = df.iloc[700:].copy()
test_df = test_df[test_df["Embarked"] != "Q"]  # category absent in this slice

encoded_test = loaded.apply_all(test_df.drop(columns=["Survived"]))
print("schema:", list(encoded_test.columns))

# 'Q' never appears here, yet the dummy column exists — filled with zeros:
encoded_test[["Embarked_S", "Embarked_C", "Embarked_Q"]].sum()

## 12. Bonus: feed it straight into a model

Sanity check that the transformed frame is genuinely model-ready (needs `scikit-learn`).

In [ ]:
try:
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from sklearn.pipeline import make_pipeline
except ImportError:
    print("scikit-learn not installed — skipping this cell (%pip install scikit-learn to run it).")
else:
    X = (
        model_ready
        .drop(columns=["Survived"])
        .select_dtypes("number")
        .drop(columns=["PassengerId"])
    )
    y = model_ready["Survived"]

    pipe = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000))
    scores = cross_val_score(pipe, X, y, cv=5)
    print(f"LogisticRegression 5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

## Wrap-up

The full loop in a few lines:

```python
import sedat

report = sedat.EDAReport.create(df, target=df["Survived"])
report.profile.summary                 # what did I load? (+ outlier notes)
report.correlations.flagged_pairs      # what's redundant?
report.imputation_summary              # what's missing & how to fix it
report.outliers.summary                # which values look extreme
model_ready = report.apply_all_encodings()   # make it trainable
clean      = report.apply_all_imputations()  # close the gaps
report.to_html("eda_report.html")       # shareable artifact
plan.to_column_transformer()            # drop into an sklearn Pipeline
sedat.suggest_datetime_features(df)     # calendar features for date columns
plan.save("plan.json")                  # reproducible train/test transforms
```

See the README for the complete API (`profile_column`, `cramers_v`, `correlation_ratio`, ...).